In [0]:
# https://docs.databricks.com/aws/en/machine-learning/feature-store/train-models-with-feature-store?
# https://docs.databricks.com/aws/en/machine-learning/feature-store/concepts

In [0]:
# Notebook: 02_Feature_Engineering
from databricks import feature_store
from pyspark.sql.functions import year, month, dayofweek
import pyspark.sql.functions as F

# Assume 'prepared_df' is available from the previous step or loaded from storage
# If running standalone, re-run data loading/prep steps from Notebook 01
# Example: Load from Delta Lake if saved previously
try:
    prepared_df = spark.read.format("delta").load("/mnt/adventureworks/prepared_data2")
    print("Loaded prepared data from Delta Lake.")
except:
    print("Failed to load from Delta Lake, attempting to re-run prep...")
#     # %run ./01_Data_Ingestion # This might be needed if running notebooks independently

Loaded prepared data from Delta Lake.


In [0]:
prepared_df.show(5)

+------------+-------------------+----------+---------+---------+--------+---------+-----------+
|SalesOrderID|          OrderDate|CustomerID| SubTotal|   TaxAmt| Freight| TotalDue|primary_key|
+------------+-------------------+----------+---------+---------+--------+---------+-----------+
|       43659|2011-05-31 00:00:00|     29825|20565.621|1971.5149|616.0984|23153.234|      43659|
|       43660|2011-05-31 00:00:00|     29672|1294.2529| 124.2483| 38.8276|1457.3289|      43660|
|       43661|2011-05-31 00:00:00|     29734|32726.479|3153.7695| 985.553|  36865.8|      43661|
|       43662|2011-05-31 00:00:00|     29994| 28832.53|2775.1646|867.2389|32474.932|      43662|
|       43663|2011-05-31 00:00:00|     29565| 419.4589|  40.2681| 12.5838| 472.3108|      43663|
+------------+-------------------+----------+---------+---------+--------+---------+-----------+
only showing top 5 rows


In [0]:
# --- Feature Engineering ---
# Create some time-based features from OrderDate
features_df = prepared_df.withColumn("OrderYear", year(F.col("OrderDate"))) \
                         .withColumn("OrderMonth", month(F.col("OrderDate"))) \
                         .withColumn("OrderDayOfWeek", dayofweek(F.col("OrderDate")))

# Select only feature columns and the primary key
# Exclude the target variable ('TotalDue') and intermediate columns like 'OrderDate'
feature_cols = ["primary_key", "CustomerID", "SubTotal", "TaxAmt", "Freight", "OrderYear", "OrderMonth", "OrderDayOfWeek"]
final_features_df = features_df.select(*feature_cols)

print("Feature engineering completed.")
display(final_features_df.limit(5))

Feature engineering completed.


primary_key,CustomerID,SubTotal,TaxAmt,Freight,OrderYear,OrderMonth,OrderDayOfWeek
43659,29825,20565.621,1971.5149,616.0984,2011,5,3
43660,29672,1294.2529,124.2483,38.8276,2011,5,3
43661,29734,32726.479,3153.7695,985.553,2011,5,3
43662,29994,28832.53,2775.1646,867.2389,2011,5,3
43663,29565,419.4589,40.2681,12.5838,2011,5,3


In [0]:
%python
# --- Feature Store ---
fs = feature_store.FeatureStoreClient()

# Define the Feature Store table name
fs_table_name = "databricks_us.adventureworks_db.sales_order_features2" # Use schema.tableName format (create schema if needed in Databricks Data Explorer)

# Create or overwrite the Feature Table
# Key 'primary_key' links features back to the original data row
try:
    fs.create_table(
        name=fs_table_name,
        primary_keys="primary_key",
        df=final_features_df,
        description="Features derived from SalesOrderHeader for predicting TotalDue."
    )
    print(f"Feature table '{fs_table_name}' created/overwritten successfully.")
except Exception as e:
    print(f"Exception occurred: {e}")
    if "already exists" in str(e).lower():
        print(f"Feature table '{fs_table_name}' already exists. Overwriting...")
        try:
            fs.write_table(
                name=fs_table_name,
                df=final_features_df,
                mode="overwrite"
            )
            print(f"Feature table '{fs_table_name}' overwritten successfully.")
        except Exception as write_e:
            print(f"Error overwriting Feature Store table: {write_e}")
            dbutils.notebook.exit("Feature Store operation failed")
    else:
        print(f"Error interacting with Feature Store: {e}")
        dbutils.notebook.exit("Feature Store operation failed")

# Example: Reading from Feature Store (optional verification)
# loaded_features = fs.read_table(name=fs_table_name)
# display(loaded_features.limit(5))

dbutils.notebook.exit(fs_table_name)